# Bluestock Capstone Project: Fund Performance Analytics

This notebook computes advanced performance and risk metrics for all 40 mutual fund schemes in the dataset, evaluates them against major Indian equity benchmarks (Nifty 50 and Nifty 100), and constructs a composite fund ranking scorecard.

### Objectives:
1. **Daily Returns Calculation**: Compute daily returns and validate their distributions.
2. **CAGR Analysis**: Compute annualized compound growth rates (1yr, 3yr, 5yr) across all schemes.
3. **Risk-Adjusted Performance**: Calculate Sharpe and Sortino ratios vs. a 6.5% risk-free rate proxy.
4. **Regression Metrics**: Perform OLS regression vs Nifty 100 returns to extract Alpha and Beta.
5. **Maximum Drawdown**: Compute max drawdowns and identify their exact peak-to-trough-to-recovery date ranges.
6. **Composite Scorecard**: Build a composite score (0-100) based on return, risk, alpha, expense, and drawdown ranks.
7. **Benchmark Comparison & Tracking Error**: Plot the top 5 funds vs. Nifty 50 & Nifty 100, and compute annualized tracking errors.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

db_path = "../db/bluestock_mf.db"
processed_dir = "../data/processed"
print("Libraries loaded successfully.")

## 1. Load Data from SQLite Database

In [ ]:
conn = sqlite3.connect(db_path)
df_funds = pd.read_sql_query("SELECT * FROM dim_fund", conn)
df_nav = pd.read_sql_query("SELECT * FROM fact_nav", conn)
df_bench = pd.read_sql_query("SELECT * FROM clean_benchmark_indices", conn)
conn.close()

# Parse dates
df_nav["date"] = pd.to_datetime(df_nav["date"])
df_bench["date"] = pd.to_datetime(df_bench["date"])

print(f"Loaded {len(df_funds)} funds.")
print(f"Loaded {len(df_nav)} NAV history records.")
print(f"Loaded {len(df_bench)} benchmark index records.")

## 2. Daily Returns Calculation & Distribution Validation

We calculate the daily returns for each scheme using: 
$$\text{daily\_return}_t = \frac{\text{NAV}_t}{\text{NAV}_{t-1}} - 1$$
And validate their distributions using descriptive statistics (Skewness and Kurtosis).

In [ ]:
df_nav.sort_values(by=["amfi_code", "date"], inplace=True)
df_nav["daily_return"] = df_nav.groupby("amfi_code")["nav"].pct_change().fillna(0.0)

# Select a representative fund for visualization
sample_fund = df_funds.iloc[0]
sample_amfi = sample_fund["amfi_code"]
sample_name = sample_fund["scheme_name"]
sample_returns = df_nav[df_nav["amfi_code"] == sample_amfi]["daily_return"]

print(f"Validation stats for fund: {sample_name} (AMFI: {sample_amfi})")
print(f"  Mean Daily Return:  {sample_returns.mean():.6f}")
print(f"  Daily Volatility:   {sample_returns.std():.6f}")
print(f"  Skewness:           {stats.skew(sample_returns):.4f} (typical equity exhibits slight negative/positive skew)")
print(f"  Kurtosis (Excess):  {stats.kurtosis(sample_returns):.4f} (high kurtosis indicates fat tails / extreme market days)")

# Plot histogram
plt.figure(figsize=(10, 5))
sns.histplot(sample_returns, bins=50, kde=True, color="#3b82f6")
plt.title(f"Daily Returns Distribution - {sample_name.split(' - ')[0]}", fontweight="bold")
plt.xlabel("Daily Return")
plt.ylabel("Frequency")
plt.show()

## 3. CAGR Calculation (1-Year, 3-Year, 5-Year)

CAGR is calculated as:
$$\text{CAGR} = \left( \frac{\text{NAV}_{\text{end}}}{\text{NAV}_{\text{start}}} \right)^{\frac{1}{n}} - 1$$
where $n$ is the period length in years.

In [ ]:
cagr_list = []

for idx, fund in df_funds.iterrows():
    amfi = fund["amfi_code"]
    fund_nav = df_nav[df_nav["amfi_code"] == amfi].sort_values("date").copy()
    if len(fund_nav) < 10:
        continue
        
    nav_end = fund_nav.iloc[-1]["nav"]
    date_end = fund_nav.iloc[-1]["date"]
    
    # 1 Year CAGR
    date_1yr = date_end - pd.DateOffset(years=1)
    nav_1yr_df = fund_nav[fund_nav["date"] <= date_1yr]
    nav_start_1yr = nav_1yr_df.iloc[-1]["nav"] if not nav_1yr_df.empty else fund_nav.iloc[0]["nav"]
    cagr_1yr = (nav_end / nav_start_1yr) - 1.0
    
    # 3 Year CAGR
    date_3yr = date_end - pd.DateOffset(years=3)
    nav_3yr_df = fund_nav[fund_nav["date"] <= date_3yr]
    nav_start_3yr = nav_3yr_df.iloc[-1]["nav"] if not nav_3yr_df.empty else fund_nav.iloc[0]["nav"]
    cagr_3yr = (nav_end / nav_start_3yr) ** (1.0 / 3.0) - 1.0
    
    # 5 Year CAGR
    nav_start_5yr = fund_nav.iloc[0]["nav"]
    date_start_5yr = fund_nav.iloc[0]["date"]
    years_diff = (date_end - date_start_5yr).days / 365.25
    cagr_5yr = (nav_end / nav_start_5yr) ** (1.0 / years_diff) - 1.0
    
    cagr_list.append({
        "amfi_code": amfi,
        "scheme_name": fund["scheme_name"],
        "fund_house": fund["fund_house"],
        "cagr_1yr_pct": cagr_1yr * 100.0,
        "cagr_3yr_pct": cagr_3yr * 100.0,
        "cagr_5yr_pct": cagr_5yr * 100.0
    })

df_cagr = pd.DataFrame(cagr_list)
print("CAGRs computed for all schemes.")
display(df_cagr.head(10))

## 4. Risk-Adjusted Ratios (Sharpe and Sortino)

We implement Sharpe and Sortino ratios using the daily risk-free rate derived from a **6.5% annual risk-free rate proxy** (RBI repo rate proxy):

$$\text{Sharpe Ratio} = \frac{R_p - R_f}{\sigma_p \times \sqrt{252}}$$

$$\text{Sortino Ratio} = \frac{R_p - R_f}{\sigma_{downside} \times \sqrt{252}}$$
where $\sigma_{downside}$ includes negative return days only.

In [ ]:
RF_RATE = 0.065
ratios_list = []

for amfi in df_cagr["amfi_code"]:
    fund_nav = df_nav[df_nav["amfi_code"] == amfi].sort_values("date").copy()
    daily_returns = fund_nav["daily_return"]
    n_days = len(fund_nav)
    
    # Annualised return (compounded daily)
    ann_return = (1.0 + daily_returns).prod() ** (252.0 / n_days) - 1.0
    
    # Annualised volatility
    std_dev_ann = daily_returns.std() * np.sqrt(252.0)
    
    # Sharpe
    sharpe = (ann_return - RF_RATE) / std_dev_ann if std_dev_ann > 0.0 else 0.0
    
    # Downside Volatility & Sortino
    negative_returns = daily_returns[daily_returns < 0.0]
    downside_std_ann = negative_returns.std() * np.sqrt(252.0)
    sortino = (ann_return - RF_RATE) / downside_std_ann if downside_std_ann > 0.0 else 0.0
    
    ratios_list.append({
        "amfi_code": amfi,
        "annualised_return_pct": ann_return * 100.0,
        "std_dev_ann_pct": std_dev_ann * 100.0,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino
    })

df_ratios = pd.DataFrame(ratios_list)
display(df_ratios.head(10))

## 5. Alpha & Beta Regression Analysis

We regress daily fund returns on daily Nifty 100 returns to compute the fund's **Beta** (systematic risk) and **Alpha** (excess return relative to benchmark index):
$$\text{daily\_return}_{\text{fund}} = \alpha_{\text{daily}} + \beta \times \text{daily\_return}_{\text{Nifty100}} + \epsilon$$
Annualized Alpha is calculated as $\alpha = \alpha_{\text{daily}} \times 252$.

In [ ]:
# Precompute benchmark daily returns
df_bench.sort_values(by=["index_name", "date"], inplace=True)
df_bench["daily_return"] = df_bench.groupby("index_name")["close_value"].pct_change().fillna(0.0)
bench_pivot = df_bench.pivot(index="date", columns="index_name", values="daily_return").fillna(0.0)

reg_list = []

for amfi in df_cagr["amfi_code"]:
    fund_nav = df_nav[df_nav["amfi_code"] == amfi].sort_values("date").copy()
    aligned = pd.merge(fund_nav[["date", "daily_return"]], bench_pivot[["NIFTY100"]], left_on="date", right_index=True)
    
    if len(aligned) > 10:
        slope, intercept, r_val, p_val, std_err = stats.linregress(aligned["NIFTY100"], aligned["daily_return"])
        beta = slope
        alpha_ann = intercept * 252.0
        r_squared = r_val ** 2
    else:
        beta = 1.0
        alpha_ann = 0.0
        r_squared = 0.0
        
    reg_list.append({
        "amfi_code": amfi,
        "alpha_nifty100_pct": alpha_ann * 100.0,
        "beta_nifty100": beta,
        "r_squared_nifty100": r_squared
    })

df_reg = pd.DataFrame(reg_list)
display(df_reg.head(10))

## 6. Maximum Drawdown and Date Ranges

We calculate the maximum drawdown for each fund, and write an algorithm to trace the **worst drawdown date range (Peak Date, Trough Date, and Recovery Date)**.

In [ ]:
dd_list = []

for amfi in df_cagr["amfi_code"]:
    fund_nav = df_nav[df_nav["amfi_code"] == amfi].sort_values("date").copy()
    fund_nav["running_max"] = fund_nav["nav"].cummax()
    fund_nav["drawdown"] = (fund_nav["nav"] - fund_nav["running_max"]) / fund_nav["running_max"]
    max_dd = fund_nav["drawdown"].min()
    
    if not fund_nav.empty and max_dd < 0:
        trough_idx = fund_nav["drawdown"].idxmin()
        trough_row = fund_nav.loc[trough_idx]
        trough_date = trough_row["date"].strftime('%Y-%m-%d')
        
        # Peak before trough
        peak_val = fund_nav.loc[trough_idx, "running_max"]
        peak_df = fund_nav[(fund_nav["date"] <= trough_row["date"]) & (fund_nav["nav"] == peak_val)]
        peak_date = peak_df.iloc[-1]["date"].strftime('%Y-%m-%d') if not peak_df.empty else fund_nav.iloc[0]["date"].strftime('%Y-%m-%d')
        
        # Recovery after trough
        recovery_df = fund_nav[(fund_nav["date"] > trough_row["date"]) & (fund_nav["nav"] >= peak_val)]
        recovery_date = recovery_df.iloc[0]["date"].strftime('%Y-%m-%d') if not recovery_df.empty else "Not Recovered"
    else:
        peak_date = "N/A"
        trough_date = "N/A"
        recovery_date = "N/A"
        
    dd_list.append({
        "amfi_code": amfi,
        "max_drawdown_pct": max_dd * 100.0,
        "worst_dd_peak_date": peak_date,
        "worst_dd_trough_date": trough_date,
        "worst_dd_recovery_date": recovery_date
    })

df_dd = pd.DataFrame(dd_list)
display(df_dd.head(10))

## 7. Composite Fund Performance Scorecard (0-100)

We construct a unified scorecard ranking all funds using the composite scoring weights:
- **30%**: 3-Year CAGR Return Rank
- **25%**: Sharpe Ratio Rank
- **20%**: Alpha vs Nifty 100 Rank
- **15%**: Expense Ratio Rank (Inverse)
- **10%**: Max Drawdown Rank (Inverse)

In [ ]:
# Merge all metrics
df_metrics = df_cagr.merge(df_ratios, on="amfi_code")
df_metrics = df_metrics.merge(df_reg, on="amfi_code")
df_metrics = df_metrics.merge(df_dd, on="amfi_code")

# Merge expense ratios from fund master
expense_map = dict(zip(df_funds["amfi_code"], df_funds["expense_ratio_pct"]))
df_metrics["expense_ratio_pct"] = df_metrics["amfi_code"].map(expense_map)

# Generate percentile ranks
df_metrics["rank_return"] = df_metrics["cagr_3yr_pct"].rank(pct=True) * 100.0
df_metrics["rank_sharpe"] = df_metrics["sharpe_ratio"].rank(pct=True) * 100.0
df_metrics["rank_alpha"] = df_metrics["alpha_nifty100_pct"].rank(pct=True) * 100.0
df_metrics["rank_expense"] = df_metrics["expense_ratio_pct"].rank(ascending=False, pct=True) * 100.0
df_metrics["rank_drawdown"] = df_metrics["max_drawdown_pct"].rank(ascending=True, pct=True) * 100.0

# Compute composite score
df_metrics["composite_score"] = (
    0.30 * df_metrics["rank_return"] +
    0.25 * df_metrics["rank_sharpe"] +
    0.20 * df_metrics["rank_alpha"] +
    0.15 * df_metrics["rank_expense"] +
    0.10 * df_metrics["rank_drawdown"]
)

df_scorecard = df_metrics.sort_values(by="composite_score", ascending=False).copy()
print("Final Fund Scorecard (Top 15 Funds):")
display(df_scorecard[["scheme_name", "cagr_3yr_pct", "sharpe_ratio", "alpha_nifty100_pct", "max_drawdown_pct", "composite_score"]].head(15))

## 8. Benchmark Comparison Chart & Tracking Error

We plot the normalized historical performance of the **Top 5 funds** vs Nifty 50 and Nifty 100 over the past 3 years. We also compute the daily **Tracking Error**:
$$\text{Tracking Error} = \text{Std}(R_{\text{fund}} - R_{\text{benchmark}}) \times \sqrt{252}$$

In [ ]:
top_5 = df_scorecard.head(5)
top_5_amfi = top_5["amfi_code"].tolist()
top_5_legends = [name.split(" - ")[0] for name in top_5["scheme_name"]]

end_date = df_nav["date"].max()
start_date = end_date - pd.DateOffset(years=3)

# Filter indices
df_bench_3y = df_bench[(df_bench["date"] >= start_date) & (df_bench["date"] <= end_date)].copy()
bench_pivot_3y = df_bench_3y.pivot(index="date", columns="index_name", values="close_value")
nifty50_s = bench_pivot_3y["NIFTY50"].dropna().sort_index()
nifty100_s = bench_pivot_3y["NIFTY100"].dropna().sort_index()

nifty50_returns = nifty50_s.pct_change().fillna(0.0)
nifty100_returns = nifty100_s.pct_change().fillna(0.0)

# Normalize indices
nifty50_norm = (nifty50_s / nifty50_s.iloc[0]) * 100.0
nifty100_norm = (nifty100_s / nifty100_s.iloc[0]) * 100.0

plt.figure(figsize=(14, 8))
plt.plot(nifty50_norm.index, nifty50_norm.values, label="NIFTY 50 (Benchmark)", color="#2563eb", linewidth=2.5, linestyle="--")
plt.plot(nifty100_norm.index, nifty100_norm.values, label="NIFTY 100 (Benchmark)", color="#4b5563", linewidth=2.5, linestyle=":")

te_results = []

for amfi, legend in zip(top_5_amfi, top_5_legends):
    fund_nav_3y = df_nav[(df_nav["amfi_code"] == amfi) & (df_nav["date"] >= start_date) & (df_nav["date"] <= end_date)].sort_values("date")
    if fund_nav_3y.empty:
        continue
        
    fund_series = fund_nav_3y.set_index("date")["nav"]
    fund_norm = (fund_series / fund_series.iloc[0]) * 100.0
    
    plt.plot(fund_norm.index, fund_norm.values, label=legend, linewidth=2.0)
    
    # Returns for tracking error
    fund_returns = fund_series.pct_change().fillna(0.0)
    
    # Align daily returns
    aligned_50 = pd.merge(fund_returns.to_frame("f"), nifty50_returns.to_frame("b"), left_index=True, right_index=True)
    aligned_100 = pd.merge(fund_returns.to_frame("f"), nifty100_returns.to_frame("b"), left_index=True, right_index=True)
    
    te_n50 = np.std(aligned_50["f"] - aligned_50["b"]) * np.sqrt(252) * 100.0
    te_n100 = np.std(aligned_100["f"] - aligned_100["b"]) * np.sqrt(252) * 100.0
    
    te_results.append({
        "Fund Name": legend,
        "3yr Return (CAGR) %": top_5[top_5["amfi_code"] == amfi].iloc[0]["cagr_3yr_pct"],
        "Sharpe Ratio": top_5[top_5["amfi_code"] == amfi].iloc[0]["sharpe_ratio"],
        "Tracking Error vs Nifty 50 (%)": te_n50,
        "Tracking Error vs Nifty 100 (%)": te_n100
    })

plt.title("Top 5 Ranked Funds vs. Benchmarks (3-Year Historical Growth)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Date", fontsize=12, labelpad=10)
plt.ylabel("Normalized Return (Start = 100)", fontsize=12, labelpad=10)
plt.legend(loc="upper left", frameon=True, fontsize=10)
plt.tight_layout()
plt.show()

print("Tracking Errors & Top 5 Performance Summary:")
display(pd.DataFrame(te_results))